<a href="https://colab.research.google.com/github/neuroneural/brainchop/blob/master/py2tfjs/Convert_Trained_Model_To_TFJS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Pytorch Model Conversion to tfjs

.

In [1]:
!wget --no-cache --backups=1 {f"https://raw.githubusercontent.com/neuroneural/brainchop/master/py2tfjs/conversion_example/blendbatchnorm.py"}
!wget --no-cache --backups=1 {f"https://raw.githubusercontent.com/neuroneural/brainchop/master/py2tfjs/conversion_example/meshnet.py"}
!wget --no-cache --backups=1 {f"https://raw.githubusercontent.com/neuroneural/brainchop/master/py2tfjs/conversion_example/meshnet2tfjs.py"}


'wget' is not recognized as an internal or external command,
operable program or batch file.


'wget' is not recognized as an internal or external command,
operable program or batch file.
'wget' is not recognized as an internal or external command,
operable program or batch file.


We'll be calling the saved trained Pytorch model to be converted to TFJS.

In [ ]:
!wget --no-cache --backups=1 {f"https://raw.githubusercontent.com/neuroneural/brainchop/master/py2tfjs/conversion_example/modelAE.json"}
!wget --no-cache --backups=1 {f"https://raw.githubusercontent.com/neuroneural/brainchop/master/py2tfjs/conversion_example/model.pth"}


In [3]:
import torch
from blendbatchnorm import fuse_bn_recursively
from meshnet2tfjs import meshnet2tfjs
from meshnet import (
    MeshNet,
    enMesh_checkpoint,
)

device_name = "cuda:0" if torch.cuda.is_available() else "cpu"
device = torch.device(device_name)


# Normalization
def preprocess_image(img, qmin=0.01, qmax=0.99):
    """Unit interval preprocessing"""
    img = (img - img.quantile(qmin)) / (img.quantile(qmax) - img.quantile(qmin))
    return img

In [ ]:
# specify how many classes does the model predict
n_classes = 3
# specify the architecture
config_file = "modelAE.json"
# how many channels does the saved model have
model_channels = 15
# path to the saved model
model_path = "model.pth"
# tfjs model output directory with colab
tfjs_model_dir = "model_tfjs"

meshnet_model = enMesh_checkpoint(
    in_channels=1,
    n_classes=n_classes,
    channels=model_channels,
    config_file=config_file,
)

checkpoint = torch.load(model_path)
meshnet_model.load_state_dict(checkpoint)

In [ ]:
meshnet_model.eval()

In [ ]:
meshnet_model.to(device)

This function takes a sequential block and fuses the batch normalization with convolution

In [ ]:
mnm = fuse_bn_recursively(meshnet_model)

del meshnet_model
mnm.model.eval()

Convert MeshNet model to TensorFlow.js

In [8]:
meshnet2tfjs(mnm, tfjs_model_dir)

In [ ]:
!ls

Save converted files to zip file

In [ ]:
!zip -r "/content/bcmodel.zip" "model_tfjs"

Download TensorFlow model

In [ ]:
from google.colab import files
files.download("/content/bcmodel.zip")

# **Final notes**

This tutorial aims to provide a simple example of how to convert segmentation model from python pipeline to TensorFlow.js files (i.e. model.json and model.bin).  